# Ask the Parks with Oracle VecDB

This notebook walks through the same APIs used by the Ask the Parks app: create a text-queryable table, load GeoJSON metadata and vectors, perform timed semantic search, optionally narrow results using spatial and metadata QBE filters, and retrieve a selected park's full record.

The QBE-ready National Park Service CSV is bundled at `data/us_national_parks_dataset_spatial.csv`. Run the setup cell only after configuring your VecDB environment variables.

In [ ]:
%pip install -U oracle-vecdb python-dotenv

In [ ]:
import os
from time import perf_counter
from dotenv import load_dotenv
from oracle_vecdb import Configuration, OracleVecDB

load_dotenv(".env", override=True)
VECDB_REST_URL = os.environ[
    "VECDB_REST_URL"
]  # must end in /_/db-api/stable/vecdb/
VECDB_TABLE = os.getenv("VECDB_TABLE", "national_parks")
VECDB_EMBED_MODEL = os.getenv("VECDB_EMBED_MODEL", "all_MiniLM_L12_v2")
SELF_SIGNED_SSL = os.getenv("VECDB_SELF_SIGNED_SSL", "false").lower() == "true"

if os.getenv("VECDB_ACCESS_TOKEN"):
    config = Configuration(
        rest_url=VECDB_REST_URL, access_token=os.environ["VECDB_ACCESS_TOKEN"]
    )
else:
    config = Configuration(
        rest_url=VECDB_REST_URL,
        username=os.environ["VECDB_USERNAME"],
        password=os.environ["VECDB_PASSWORD"],
    )

if SELF_SIGNED_SSL:
    config.verify_ssl = False

vecdb = OracleVecDB(config)


def timed_query(**kwargs):
    """Run the same query call as the app and return its server-side elapsed time."""
    started = perf_counter()
    response = vecdb.query(**kwargs)
    return response, round((perf_counter() - started) * 1000, 1)


print(f"Connected client configured for table: {VECDB_TABLE}")

## Inspect available models

The table needs the same embedding model that produced its supplied 384-dimension vectors. The loader defaults to `all_MiniLM_L12_v2`; confirm that it is available before creating the table.

In [ ]:
models = vecdb.list_models()
for model in models.items or []:
    print(
        getattr(model, "model_name", None), getattr(model, "dimensions", None)
    )

## Create and load the table (optional, writes data)

The repository loader downloads the CSV, validates `metadata.location` as GeoJSON, then calls `upsert_vectors` in batches. It is disabled below so opening the notebook never changes your database. Set `RUN_LOAD = True` for a new table, or use the command-line script described in the README.

In [ ]:
RUN_LOAD = False

if RUN_LOAD:
    from load_parks_vecdb import batches, records

    # SDK signature used by the installed client: create_vector_table(name=...).
    vecdb.create_vector_table(
        name=VECDB_TABLE,
        embed_params={
            "model": VECDB_EMBED_MODEL,
            "embed_metadata_jsonpath": "DESCRIPTION",
        },
    )

    for batch in batches(records(None, csv_file="data/us_national_parks_dataset_spatial.csv"), 50):
        # SDK signature used by the installed client: upsert_vectors(table_name=..., vectors=...).
        vecdb.upsert_vectors(table_name=VECDB_TABLE, vectors=batch)
    print("National parks loaded.")

## Inspect the table

This confirms the table and its embedding configuration before querying it.

In [ ]:
table = vecdb.describe_vector_table(name=VECDB_TABLE)
print(table)

## 1. Semantic vector search (the default flow)

`query_by={"text": ...}` asks VecDB to generate a hosted query embedding and return the most semantically relevant parks. This is the app's default behavior when location is blank. The elapsed time is what the app surfaces as search latency.

In [ ]:
semantic_results, latency_ms = timed_query(
    table_name=VECDB_TABLE,
    query_by={"text": "peaceful desert park with short hikes"},
    top_k=5,
)
print(f"Search latency: {latency_ms} ms")

for item in semantic_results.items or []:
    metadata = item.metadata
    print(
        metadata.get("NAME"),
        "—",
        metadata.get("STATES"),
        "score:",
        getattr(item, "score", None),
    )

## 2. Optionally add a spatial QBE filter

Location is not a separate search mode. It is an optional QBE filter combined with the semantic query. GeoJSON coordinates use `[longitude, latitude]`; omit `filters` entirely to reproduce **Clear location** in the app.

In [ ]:
san_francisco_radius = {
    "location": {
        "$near": {
            "$geometry": {"type": "Point", "coordinates": [-122.4194, 37.7749]},
            "$distance": 500,
            "$unit": "KM",
        }
    }
}

nearby_results, nearby_latency_ms = timed_query(
    table_name=VECDB_TABLE,
    query_by={"text": "peaceful desert park with short hikes"},
    top_k=10,
    filters=san_francisco_radius,
)
print(f"Spatial search latency: {nearby_latency_ms} ms")

for item in nearby_results.items or []:
    print(item.metadata.get("NAME"), item.metadata.get("location"))

## 3. Combine spatial and metadata QBE

Use `$and` to combine the optional radius with another metadata condition. This pattern is what the application uses when a user enters both a city and a Metadata QBE filter. The app's filter builder also generates `$regex` for its Contains and Matches regex options.

In [ ]:
filters = {
    "$and": [
        san_francisco_radius,
        {"DESIGNATION": {"$upper": {"$startsWith": "NATIONAL PARK"}}},
    ]
}

filtered_results, filtered_latency_ms = timed_query(
    table_name=VECDB_TABLE,
    query_by={"text": "family-friendly scenic hiking"},
    top_k=10,
    filters=filters,
)
print(f"Combined-filter latency: {filtered_latency_ms} ms")

for item in filtered_results.items or []:
    print(item.metadata.get("FULL_NAME"), "|", item.metadata.get("STATES"))

# This is the QBE generated by a Park name "Matches regex" filter in the app.
adams_name_regex = {"NAME": {"$regex": ".*Adams.*"}}
regex_results, regex_latency_ms = timed_query(
    table_name=VECDB_TABLE,
    query_by={"text": "historic park"},
    top_k=5,
    filters=adams_name_regex,
)
print(f"Regex-filter latency: {regex_latency_ms} ms")
for item in regex_results.items or []:
    print(item.metadata.get("FULL_NAME"))

## 4. Fetch the selected park's full record

Cards and map markers show a concise search result. When a user selects one, the app calls `list_vectors()` with its id so the details drawer can display all stored metadata, such as directions and weather.

In [ ]:
matches = filtered_results.items or semantic_results.items or []
if not matches:
    raise RuntimeError(
        "No park was returned; widen the radius or change the query."
    )

selected_id = matches[0].id
detail_response = vecdb.list_vectors(table_name=VECDB_TABLE, ids=[selected_id])
selected = (detail_response.items or [])[0]
metadata = selected.metadata

print(metadata.get("FULL_NAME"))
print("Directions:", metadata.get("DIRECTIONS_INFO", "Not supplied"))
print("Weather:", metadata.get("WEATHER_INFO", "Not supplied"))

## Next: run the app

With the same `VECDB_*` environment variables, start `python3 app.py`. The interface calls the same `query()` API, shows server-side latency, lets developers add or clear an optional city/radius filter, accepts raw Metadata QBE, and links result cards, map markers, and the full `list_vectors()` details drawer.